# よくある質問に自動で答える FAQ ボットを作る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_dialogue/faq_bot.ipynb)

NLP クックブック 22-02 ／ 対話システム

用意した「質問と回答」の一覧から、ユーザーの質問に最も近いものを探して回答を返したい。LLM に生成させず、決まった回答だけを返したい。

In [1]:
from sentence_transformers import SentenceTransformer

faq = [
    {"q": ["返品はできますか", "商品を返したい", "返品の期限はいつまで"], "a": "商品到着から 14 日以内であれば返品できます。マイページの注文履歴からお手続きください。"},
    {"q": ["送料はいくらですか", "配送料を教えて", "送料無料になる条件は"], "a": "送料は全国一律 550 円です。5,000 円以上のご注文で無料になります。"},
    {"q": ["支払い方法は何がありますか", "クレジットカードは使えますか", "コンビニ払いはできますか"], "a": "クレジットカード、コンビニ払い、銀行振込がご利用いただけます。"},
    {"q": ["注文をキャンセルしたい", "キャンセルできますか"], "a": "発送前であれば、マイページの注文履歴からキャンセルできます。"},
    {"q": ["届くまで何日かかりますか", "いつ届きますか", "配送状況を知りたい"], "a": "通常、ご注文から 2〜4 日でお届けします。発送後にお送りするメールの追跡番号で配送状況を確認できます。"},
    {"q": ["パスワードを忘れました", "ログインできない"], "a": "ログイン画面の「パスワードをお忘れの方」から再設定できます。"},
]
model = SentenceTransformer("intfloat/multilingual-e5-small")
questions = [(i, q) for i, item in enumerate(faq) for q in item["q"]]      # 1 つの回答に複数の言い回しを登録しておく
q_emb = model.encode(["query: " + q for _, q in questions], normalize_embeddings=True)

def answer(text, threshold=0.88, margin=0.01):
    sims = q_emb @ model.encode("query: " + text, normalize_embeddings=True)
    best = {}
    for (i, _), s in zip(questions, sims):                 # 回答ごとに、登録した言い回しの中の最大類似度をとる
        best[i] = max(best.get(i, 0), float(s))
    ranked = sorted(best.items(), key=lambda x: -x[1])
    (top, s1), (second, s2) = ranked[0], ranked[1]
    if s1 < threshold:
        return f"({s1:.2f}) 申し訳ありません。担当者におつなぎします。"
    if s1 - s2 < margin:                                    # 上位 2 つが僅差なら、どちらのことか聞き返す
        return f"({s1:.2f}/{s2:.2f}) 「{faq[top]['q'][0]}」と「{faq[second]['q'][0]}」のどちらについてのご質問でしょうか？"
    return f"({s1:.2f}) {faq[top]['a']}"

for text in ["買ったものを返したいんだけど", "何日くらいで届く？", "カードで払える？", "ログインのパスワードがわからなくなった",
             "注文したけどやっぱりやめたい", "ギフト包装はできますか？", "今日の天気は？"]:
    print(f"Q: {text}\n   {answer(text)}")

Q: 買ったものを返したいんだけど
   (0.96) 商品到着から 14 日以内であれば返品できます。マイページの注文履歴からお手続きください。
Q: 何日くらいで届く？
   (0.98) 通常、ご注文から 2〜4 日でお届けします。発送後にお送りするメールの追跡番号で配送状況を確認できます。
Q: カードで払える？
   (0.91) クレジットカード、コンビニ払い、銀行振込がご利用いただけます。
Q: ログインのパスワードがわからなくなった
   (0.95) ログイン画面の「パスワードをお忘れの方」から再設定できます。
Q: 注文したけどやっぱりやめたい
   (0.92) 発送前であれば、マイページの注文履歴からキャンセルできます。
Q: ギフト包装はできますか？
   (0.84) 申し訳ありません。担当者におつなぎします。
Q: 今日の天気は？
   (0.81) 申し訳ありません。担当者におつなぎします。


- 回答文は人が書いたものをそのまま返すので、誤ったことを言う心配がない。問い合わせ対応では、この確実さが生成より重視されることが多い。
- 1 つの回答に対して、言い回しの違う質問を複数登録しておくと当たりやすくなる。当たらなかった質問のログから、言い回しを足していく。
- しきい値未満は「該当なし」として人に引き継ぐ。無理に一番近い回答を返すと、的外れな案内になる。
- FAQ にない質問にも答えたい場合は、RAG の章のように文書を検索して LLM に答えさせる方法と組み合わせる。